Notebook セル 1: 環境と入出力パス定義

In [ ]:
from pathlib import Path
import sys, importlib, pandas as pd

repo = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project")
src_dir = repo / "src"
data_dir = repo / "data"

# import path
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# 入出力
csv_path = data_dir / r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\resources\point_list_sample#3.csv"  # 5列スキーマ版
out_ttl  = data_dir / r"Out\points_from_csv#.ttl"

print("CSV exists:", csv_path.exists())
print("Out dir:   ", out_ttl.parent)
out_ttl.parent.mkdir(parents=True, exist_ok=True)

CSV exists: True
Out dir:    C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\Out


Notebook セル 2: ローダの読み込みと Brick 生成

In [ ]:
from points_csv_ingestor import loader
importlib.reload(loader)

g = loader.csv_to_brick_graph(str(csv_path), verbose=True)
loader.write_graph(g, str(out_ttl))
print("TTL written:", out_ttl.exists(), "triples:", len(g))

Notebook セル 3: グラフ統計の基礎検証

In [ ]:
from rdflib import Namespace

BRICK = Namespace("https://brickschema.org/schema/Brick#")
BOT   = Namespace("https://w3id.org/bot#")

spaces = sum(1 for _ in g.triples((None, loader.RDF.type, BOT.Space)))
sensors = sum(1 for s,p,o in g.triples((None, loader.RDF.type, None)) if o != BOT.Space)
locs = sum(1 for _ in g.triples((None, BRICK.hasLocation, None)))
units = sum(1 for _ in g.triples((None, BRICK.hasUnit, None)))

print("Spaces:", spaces)
print("Sensors:", sensors)
print("hasLocation:", locs)
print("hasUnit:", units)

Notebook セル 4: hasLocation のサンプル列挙

In [ ]:
for i, (s, p, o) in enumerate(g.triples((None, BRICK.hasLocation, None))):
    if i >= 10:
        break
    print(f"{i+1:02d}. {s} -> {o}")

Notebook セル 5: Occupancy センサーの単位が付いていないことの確認

In [ ]:
from rdflib import URIRef

occ = BRICK["Occupancy_Sensor"]
has_unit = BRICK["hasUnit"]

cnt_occ = 0
cnt_occ_with_unit = 0
for s, _, o in g.triples((None, loader.RDF.type, occ)):
    cnt_occ += 1
    if any(True for _ in g.triples((s, has_unit, None))):
        cnt_occ_with_unit += 1

print("Occupancy sensors:", cnt_occ)
print("Occupancy sensors with unit:", cnt_occ_with_unit, "(should be 0 if unit=binary)")

Notebook セル 6: CSV 未充填や不一致の監査

In [ ]:
df = pd.read_csv(csv_path, dtype=object)

# bdns が欠損かつ space_id 欠損の行
mask_bdns_missing = df["bdns_abbreviation"].isna() | df["bdns_abbreviation"].eq("")
mask_space_missing = df["space_id"].isna() | df["space_id"].eq("")

todo = df[mask_bdns_missing & mask_space_missing]
print("Rows require space_id but missing:", len(todo))
display(todo.head(10))